# Mangrove Pulse: cached dated-context replay

Prepared 8 October 2026. This notebook reads the exact small crop retained from the successful 24 September 2026 SCL/TCI context experiment, without requesting imagery or downloading inputs again. The [dated-context guide](../docs/DATED_EO_CONTEXT.md) records the replay result and its limits.

Contains modified Copernicus Sentinel data (2026), accessed through Microsoft Planetary Computer. [Source/modification notice](https://cds.climate.copernicus.eu/licences/ec-sentinel).

The 2018 experiment remains frozen. Current product labels are not independent historical validation, mangrove field truth, chemistry, navigability, drone superiority or ecological benefit. No sampling stations or field observations exist.


## 1. Restore exact cached inputs

The reader uses the retained cache and NumPy already present in standard Colab. If NumPy is unavailable, it fails without installing anything. Embedded hashes protect the exact source, report and cache.


In [ ]:
# No pip install, Drive mount, upload picker or imagery request.
import base64, hashlib, json, sys, tempfile
from pathlib import Path
PAYLOADS = json.loads("{\"offline_context_reader.py\":{\"b64\":\"IiIiUmVwbGF5IHNhdmVkIHNjZW5lLWxhYmVsIHN1bW1hcmllcyBpbiBDb2xhYiwgd2l0aCBubyByYXN0ZXIvbmV0d29yayBhY3F1aXNpdGlvbi4KClRoaXMgY2hlY2tzIGNhY2hlIGludGVncml0eSBhbmQgcmVwZWF0cyBkZXNjcmlwdGl2ZSBjb3VudHMuIEl0IGlzIG5vdCBpbmRlcGVuZGVudAp2YWxpZGF0aW9uIG9mIHRoZSBzb3VyY2UgcHJvZHVjdCwgQU9JIG1lbWJlcnNoaXAsIGZpZWxkIGNvdmVyIG9yIGRyb25lIGNsYWltcy4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCBpbwppbXBvcnQganNvbgoKQ0FDSEVfU0hBMjU2ID0gIjFmODllZmY2OWRiOTBmNmNlY2MyMDIxY2RiZWIwOTBiYzIzZDFmNDA1OWQ0M2MyMDYzOTI2ZWI5NDUzZGY1ODkiClJFUE9SVF9TSEEyNTYgPSAiYWM5NTE1OWI5MDE5NmNhM2ZiNDdiY2QwZjQzYWVkZjkyYWI3M2EzZmM1YzM5MjE5MjQ2YjVlMTRkNDc3ZDYxMSIKSVRFTV9JRCA9ICJTMkNfTVNJTDJBXzIwMjYwOTI0VDA2NDYyMV9SMDIwX1Q0MFJETl8yMDI2MDkyNFQxMTQ3MTYiCgoKZGVmIHJ1bihjYWNoZV9ieXRlczogYnl0ZXMsIHJlcG9ydF9ieXRlczogYnl0ZXMpIC0+IGRpY3Q6CiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICBjaGVja3MgPSBbXQoKICAgIGRlZiByZXF1aXJlKGNvbmRpdGlvbiwgbGFiZWwpOgogICAgICAgIGlmIG5vdCBjb25kaXRpb246CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IobGFiZWwpCiAgICAgICAgY2hlY2tzLmFwcGVuZChsYWJlbCkKCiAgICByZXF1aXJlKGhhc2hsaWIuc2hhMjU2KGNhY2hlX2J5dGVzKS5oZXhkaWdlc3QoKSA9PSBDQUNIRV9TSEEyNTYsICJFeGFjdCByZXRhaW5lZCBjcm9wIGhhc2giKQogICAgcmVxdWlyZShoYXNobGliLnNoYTI1NihyZXBvcnRfYnl0ZXMpLmhleGRpZ2VzdCgpID09IFJFUE9SVF9TSEEyNTYsICJFeGFjdCByZXR1cm5lZCByZXBvcnQgaGFzaCIpCiAgICByZXBvcnQgPSBqc29uLmxvYWRzKHJlcG9ydF9ieXRlcykKICAgIHJlcXVpcmUocmVwb3J0WyJpdGVtX2lkIl0gPT0gSVRFTV9JRCBhbmQgcmVwb3J0WyJhY3F1aXNpdGlvbl9kYXRlIl0gPT0gIjIwMjYtMDktMjQiLAogICAgICAgICAgICAiRXhwZWN0ZWQgaXRlbSBhbmQgZGF0ZSIpCiAgICByZXF1aXJlKHJlcG9ydFsiZmllbGRfb2JzZXJ2YXRpb25zIl0gPT0gcmVwb3J0WyJpbmRlcGVuZGVudF8yMDE4X2xhYmVscyJdID09IDAKICAgICAgICAgICAgYW5kIHJlcG9ydFsic2FtcGxpbmdfc3RhdGlvbnMiXSA9PSBbXSBhbmQgcmVwb3J0WyJkZXBsb3ltZW50X2F1dGhvcml6ZWQiXSBpcyBGYWxzZSwKICAgICAgICAgICAgIk5vIGVtcGlyaWNhbCBldmlkZW5jZSBvciBkZXBsb3ltZW50IGF1dGhvcml6YXRpb24iKQogICAgcmVxdWlyZShyZXBvcnRbInNwZWN0cmFsX2luZGljZXNfY29tcHV0ZWQiXSA9PSBbXSBhbmQgcmVwb3J0WyJyZWZsZWN0YW5jZV9iYW5kc191c2VkIl0gPT0gW10sCiAgICAgICAgICAgICJObyBjYWxpYnJhdGVkIHNwZWN0cmFsIGFuYWx5c2lzIikKICAgIHNjaGVtYSA9IHsKICAgICAgICAic2NsIjogKCgxMTIsIDEyMiksICJ1aW50OCIpLCAic2NsX21hc2siOiAoKDExMiwgMTIyKSwgImJvb2wiKSwKICAgICAgICAic2NsX2luc2lkZSI6ICgoMTEyLCAxMjIpLCAiYm9vbCIpLCAidGNpIjogKCgzLCAyMjMsIDI0NCksICJ1aW50OCIpLAogICAgICAgICJ0Y2lfbWFzayI6ICgoMjIzLCAyNDQpLCAiYm9vbCIpLCAidGNpX2luc2lkZSI6ICgoMjIzLCAyNDQpLCAiYm9vbCIpLAogICAgICAgICJzY2xfdHJhbnNmb3JtIjogKCg2LCksICJmbG9hdDY0IiksICJ0Y2lfdHJhbnNmb3JtIjogKCg2LCksICJmbG9hdDY0IiksCiAgICB9CiAgICB3aXRoIG5wLmxvYWQoaW8uQnl0ZXNJTyhjYWNoZV9ieXRlcyksIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgY2FjaGU6CiAgICAgICAgcmVxdWlyZShzZXQoY2FjaGUuZmlsZXMpID09IHNldChzY2hlbWEpIHwgeyJpdGVtX2lkIiwgImNycyJ9LCAiRXhhY3QgdGVuLWZpZWxkIGNhY2hlIHNjaGVtYSIpCiAgICAgICAgcmVxdWlyZShjYWNoZVsiaXRlbV9pZCJdLnNoYXBlID09ICgpIGFuZCBjYWNoZVsiaXRlbV9pZCJdLmR0eXBlLmtpbmQgPT0gIlUiCiAgICAgICAgICAgICAgICBhbmQgc3RyKGNhY2hlWyJpdGVtX2lkIl0uaXRlbSgpKSA9PSBJVEVNX0lELCAiQ2FjaGUgaXRlbSBpZGVudGl0eSIpCiAgICAgICAgcmVxdWlyZShjYWNoZVsiY3JzIl0uc2hhcGUgPT0gKCkgYW5kIGNhY2hlWyJjcnMiXS5kdHlwZS5raW5kID09ICJVIgogICAgICAgICAgICAgICAgYW5kIHN0cihjYWNoZVsiY3JzIl0uaXRlbSgpKSA9PSAiRVBTRzozMjY0MCIsICJDYWNoZSBwcm9qZWN0ZWQgQ1JTIikKICAgICAgICBmb3Iga2V5LCAoc2hhcGUsIGR0eXBlKSBpbiBzY2hlbWEuaXRlbXMoKToKICAgICAgICAgICAgcmVxdWlyZShjYWNoZVtrZXldLnNoYXBlID09IHNoYXBlIGFuZCBjYWNoZVtrZXldLmR0eXBlID09IG5wLmR0eXBlKGR0eXBlKSwKICAgICAgICAgICAgICAgICAgICAiQ2FjaGUgc2hhcGUvZHR5cGU6ICIgKyBrZXkpCiAgICAgICAgZm9yIGtleSwgcmVzb2x1dGlvbiwgcmVwb3J0X2tleSBpbiAoKCJzY2xfdHJhbnNmb3JtIiwgMjAsICJzY2wiKSwgKCJ0Y2lfdHJhbnNmb3JtIiwgMTAsICJ0Y2kiKSk6CiAgICAgICAgICAgIGFmZmluZSA9IGNhY2hlW2tleV0KICAgICAgICAgICAgcmVxdWlyZShib29sKG5wLmlzZmluaXRlKGFmZmluZSkuYWxsKCkpIGFuZCBhZmZpbmVbMF0gPT0gcmVzb2x1dGlvbgogICAgICAgICAgICAgICAgICAgIGFuZCBhZmZpbmVbNF0gPT0gLXJlc29sdXRpb24gYW5kIGFmZmluZVsxXSA9PSBhZmZpbmVbM10gPT0gMCwKICAgICAgICAgICAgICAgICAgICAiTmF0aXZlIG5vcnRoLXVwIGNyb3AgYWZmaW5lOiAiICsga2V5KQogICAgICAgICAgICBzb3VyY2UgPSByZXBvcnRbcmVwb3J0X2tleV1bInNvdXJjZV9hZmZpbmUiXQogICAgICAgICAgICBjb2wgPSBmbG9hdCgoYWZmaW5lWzJdIC0gc291cmNlWzJdKSAvIHJlc29sdXRpb24pCiAgICAgICAgICAgIHJvdyA9IGZsb2F0KChzb3VyY2VbNV0gLSBhZmZpbmVbNV0pIC8gcmVzb2x1dGlvbikKICAgICAgICAgICAgcmVxdWlyZShjb2wgPj0gMCBhbmQgcm93ID49IDAgYW5kIGNvbC5pc19pbnRlZ2VyKCkgYW5kIHJvdy5pc19pbnRlZ2VyKCksCiAgICAgICAgICAgICAgICAgICAgIkNyb3Agb3JpZ2luIGxpZXMgb24gcmVjb3JkZWQgc291cmNlIGdyaWQ6ICIgKyBrZXkpCiAgICAgICAgIyBSZXVzZSBleGFjdCBzYXZlZCBtZW1iZXJzaGlwLiBUaGlzIGRvZXMgbm90IGluZGVwZW5kZW50bHkgcmVjb21wdXRlIHRoZSBBT0kuCiAgICAgICAgc2NsLCBzb3VyY2VfbWFzaywgaW5zaWRlID0gY2FjaGVbInNjbCJdLCBjYWNoZVsic2NsX21hc2siXSwgY2FjaGVbInNjbF9pbnNpZGUiXQogICAgICAgIHZhbGlkID0gaW5zaWRlICYgfnNvdXJjZV9tYXNrICYgKHNjbCAhPSAwKQogICAgICAgIHJlcXVpcmUoYm9vbCh2YWxpZC5hbnkoKSksICJOb25lbXB0eSB2YWxpZCBTQ0wgc2VsZWN0aW9uIikKICAgICAgICByZXF1aXJlKGJvb2wobnAuYWxsKHNjbFt2YWxpZF0gPD0gMTEpKSwgIlNDTCBjbGFzcyBkb21haW4iKQogICAgICAgIGNvdW50cyA9IHtzdHIoY29kZSk6IGludChucC5jb3VudF9ub256ZXJvKHZhbGlkICYgKHNjbCA9PSBjb2RlKSkpIGZvciBjb2RlIGluIHJhbmdlKDEyKX0KICAgICAgICBtYXNrZWQgPSBpbnQobnAuY291bnRfbm9uemVybyhpbnNpZGUgJiB+dmFsaWQpKQogICAgICAgIHJlcXVpcmUoY291bnRzID09IHJlcG9ydFsic2NsIl1bInZhbGlkX2NsYXNzX2NvdW50cyJdLCAiQ2FjaGVkIHByb2R1Y3QtbGFiZWwgY291bnRzIHJlcHJvZHVjZSByZXBvcnQiKQogICAgICAgIHJlcXVpcmUobWFza2VkID09IHJlcG9ydFsic2NsIl1bIm1hc2tlZF9vcl9ub2RhdGFfY291bnQiXSwgIlNDTCBtYXNrZWQvTm9EYXRhIGNvdW50IHJlcHJvZHVjZXMgcmVwb3J0IikKICAgICAgICBjZW50cmVzID0gaW50KG5wLmNvdW50X25vbnplcm8oaW5zaWRlKSkKICAgICAgICByZXF1aXJlKGNlbnRyZXMgPT0gcmVwb3J0WyJzY2wiXVsiYW9pX2NlbGxfY2VudHJlcyJdID09IDEzNDAxCiAgICAgICAgICAgICAgICBhbmQgc3VtKGNvdW50cy52YWx1ZXMoKSkgKyBtYXNrZWQgPT0gY2VudHJlcywgIk5hdGl2ZSBTQ0wgY2VudHJlLWNvdW50IGNsb3N1cmUiKQogICAgICAgIHRtYXNrLCB0aW5zaWRlID0gY2FjaGVbInRjaV9tYXNrIl0sIGNhY2hlWyJ0Y2lfaW5zaWRlIl0KICAgICAgICByZXF1aXJlKGJvb2wobnAuYWxsKHRtYXNrW350aW5zaWRlXSkpLCAiVENJIG91dHNpZGUtQU9JIHBpeGVscyBhcmUgbWFza2VkIikKICAgICAgICByZXF1aXJlKGJvb2wobnAuYW55KHRpbnNpZGUgJiB+dG1hc2spKSwgIk5vbmVtcHR5IFRDSSBkaXNwbGF5IHNlbGVjdGlvbiIpCiAgICAgICAgcmVxdWlyZShpbnQobnAuY291bnRfbm9uemVybyh0bWFzaykpID09IHJlcG9ydFsidGNpIl1bIm1hc2tlZF9vcl9vdXRzaWRlX2FvaV9waXhlbF9jb3VudCJdLAogICAgICAgICAgICAgICAgIlRDSSBjb21iaW5lZCBtYXNrIGNvdW50IHJlcHJvZHVjZXMgcmVwb3J0IikKICAgIHJldHVybiB7CiAgICAgICAgImV4cGVyaW1lbnRfaWQiOiAiTVAtS0FMQkEtT0ZGTElORS1DT05URVhULTAxIiwKICAgICAgICAic3RhdHVzIjogInN1Y2NlZWRlZF9jYWNoZWRfc3VtbWFyeV9yZXBsYXkiLAogICAgICAgICJpdGVtX2lkIjogSVRFTV9JRCwgImFjcXVpc2l0aW9uX2RhdGUiOiAiMjAyNi0wOS0yNCIsCiAgICAgICAgImNhY2hlX3NoYTI1NiI6IENBQ0hFX1NIQTI1NiwgInJldHVybmVkX3JlcG9ydF9zaGEyNTYiOiBSRVBPUlRfU0hBMjU2LAogICAgICAgICJudW1weV92ZXJzaW9uIjogbnAuX192ZXJzaW9uX18sICJjaGVja3NfcGFzc2VkIjogbGVuKGNoZWNrcyksICJjaGVja3MiOiBjaGVja3MsCiAgICAgICAgIm5hdGl2ZV8yMG1fc2NsX2NvdW50cyI6IGNvdW50cywgImFvaV9jZW50cmVzIjogY2VudHJlcywKICAgICAgICAiZW9fbmV0d29ya19yZXF1ZXN0c19pbl9yZWFkZXIiOiAwLAogICAgICAgICJzb3VyY2VfQU9JX21lbWJlcnNoaXBfcmVjb21wdXRlZCI6IEZhbHNlLAogICAgICAgICJpbmRlcGVuZGVudF92YWxpZGF0aW9uIjogRmFsc2UsICJmaWVsZF9vYnNlcnZhdGlvbnMiOiAwLAogICAgICAgICJpbmRlcGVuZGVudF8yMDE4X2xhYmVscyI6IDAsICJzYW1wbGluZ19zdGF0aW9ucyI6IFtdLAogICAgICAgICJkZXBsb3ltZW50X2F1dGhvcml6ZWQiOiBGYWxzZSwgInJlZmxlY3RhbmNlX2luZGljZXNfY29tcHV0ZWQiOiBbXSwKICAgICAgICAibm90X2luZmVycmVkIjogcmVwb3J0WyJub3RfaW5mZXJyZWQiXSwKICAgICAgICAibGltaXRzIjogWyJSZXBsYXkgb2YgY2FjaGVkIHByb2R1Y3QgbGFiZWxzLCBub3QgZmllbGQgdHJ1dGggb3IgY3VycmVudCBtYW5ncm92ZSBleHRlbnQuIiwKICAgICAgICAgICAgICAgICAgICJObyBoaXN0b3JpY2FsIG92ZXJsYXksIGNoZW1pY2FsIG1lYXN1cmVtZW50LCBhaXJjcmFmdCB0ZXN0IG9yIGVjb2xvZ2ljYWwgb3V0Y29tZS4iLAogICAgICAgICAgICAgICAgICAgIk5ldHdvcmstZnJlZSByZWFkZXIgZG9lcyBub3QgbWVhbiB6ZXJvIGJyb3dzZXIvbm90ZWJvb2svc2F2ZSB0cmFuc2ZlciBvdmVyaGVhZC4iXSwKICAgIH0K\",\"sha256\":\"ca1c8f7f92fce2693188cbad59b445cb9d2ffb0942d8adf8569dd6f77418246a\"},\"native_context_cache.npz\":{\"b64\":\"...\",\"sha256\":\"1f89eff69db90f6cecc2021cdbeb090bc23d1f4059d43c2063926eb9453df589\"},\"class_context_report.json\":{\"b64\":\"ewogICJleHBlcmltZW50X2lkIjogIk1QLUtBTEJBLVJFQ0VOVC1DT05URVhULTAyIiwKICAic3RhdHVzIjogImNvbnRleHRfZ2VuZXJhdGVkIiwKICAiaXRlbV9pZCI6ICJTMkNfTVNJTDJBXzIwMjYwOTI0VDA2NDYyMV9SMDIwX1Q0MFJETl8yMDI2MDkyNFQxMTQ3MTYiLAogICJjb2xsZWN0aW9uIjogInNlbnRpbmVsLTItbDJhIiwKICAicHJvY2Vzc2luZ19iYXNlbGluZSI6ICIwNS4xMyIsCiAgImFjcXVpc2l0aW9uX2RhdGUiOiAiMjAyNi0wOS0yNCIsCiAgImFvaV93Z3M4NCI6IFsKICAgIDU2LjM1LAogICAgMjQuOTg2LAogICAgNTYuMzc0LAogICAgMjUuMDA2CiAgXSwKICAiYXNzZXRzIjogewogICAgIlNDTCI6IHsKICAgICAgImtleSI6ICJTQ0wiLAogICAgICAiaG9zdCI6ICJzZW50aW5lbDJsMmEwMS5ibG9iLmNvcmUud2luZG93cy5uZXQiLAogICAgICAicGF0aCI6ICIvc2VudGluZWwyLWwyLzQwL1IvRE4vMjAyNi8wOS8yNC9TMkNfTVNJTDJBXzIwMjYwOTI0VDA2NDYyMV9OMDUxM19SMDIwX1Q0MFJETl8yMDI2MDkyNFQxMTQ3MTYuU0FGRS9HUkFOVUxFL0wyQV9UNDBSRE5fQTAxMDcxNV8yMDI2MDkyNFQwNjU5NTkvSU1HX0RBVEEvUjIwbS9UNDBSRE5fMjAyNjA5MjRUMDY0NjIxX1NDTF8yMG0udGlmIiwKICAgICAgInR5cGUiOiAiaW1hZ2UvdGlmZjsgYXBwbGljYXRpb249Z2VvdGlmZjsgcHJvZmlsZT1jbG91ZC1vcHRpbWl6ZWQiLAogICAgICAicm9sZXMiOiBbCiAgICAgICAgImRhdGEiCiAgICAgIF0sCiAgICAgICJmaWxlOnNpemUiOiBudWxsLAogICAgICAiY2hlY2tzdW06bXVsdGloYXNoIjogbnVsbAogICAgfSwKICAgICJ2aXN1YWwiOiB7CiAgICAgICJrZXkiOiAidmlzdWFsIiwKICAgICAgImhvc3QiOiAic2VudGluZWwybDJhMDEuYmxvYi5jb3JlLndpbmRvd3MubmV0IiwKICAgICAgInBhdGgiOiAiL3NlbnRpbmVsMi1sMi80MC9SL0ROLzIwMjYvMDkvMjQvUzJDX01TSUwyQV8yMDI2MDkyNFQwNjQ2MjFfTjA1MTNfUjAyMF9UNDBSRE5fMjAyNjA5MjRUMTE0NzE2LlNBRkUvR1JBTlVMRS9MMkFfVDQwUkROX0EwMTA3MTVfMjAyNjA5MjRUMDY1OTU5L0lNR19EQVRBL1IxMG0vVDQwUkROXzIwMjYwOTI0VDA2NDYyMV9UQ0lfMTBtLnRpZiIsCiAgICAgICJ0eXBlIjogImltYWdlL3RpZmY7IGFwcGxpY2F0aW9uPWdlb3RpZmY7IHByb2ZpbGU9Y2xvdWQtb3B0aW1pemVkIiwKICAgICAgInJvbGVzIjogWwogICAgICAgICJkYXRhIgogICAgICBdLAogICAgICAiZmlsZTpzaXplIjogbnVsbCwKICAgICAgImNoZWNrc3VtOm11bHRpaGFzaCI6IG51bGwKICAgIH0KICB9LAogICJzY2wiOiB7CiAgICAiY3JzIjogIkVQU0c6MzI2NDAiLAogICAgIm5hdGl2ZV9yZXNvbHV0aW9uX20iOiBbCiAgICAgIDIwLjAsCiAgICAgIDIwLjAKICAgIF0sCiAgICAic291cmNlX2FmZmluZSI6IFsKICAgICAgMjAuMCwKICAgICAgMC4wLAogICAgICAzOTk5NjAuMCwKICAgICAgMC4wLAogICAgICAtMjAuMCwKICAgICAgMjgwMDAyMC4wCiAgICBdLAogICAgIm5vZGF0YV92YWx1ZSI6IG51bGwsCiAgICAid2luZG93X3NoYXBlX3Jvd3NfY29scyI6IFsKICAgICAgMTEyLAogICAgICAxMjIKICAgIF0sCiAgICAiYW9pX2NlbGxfY2VudHJlcyI6IDEzNDAxLAogICAgInZhbGlkX2NsYXNzX2NvdW50cyI6IHsKICAgICAgIjAiOiAwLAogICAgICAiMSI6IDAsCiAgICAgICIyIjogMCwKICAgICAgIjMiOiAxMiwKICAgICAgIjQiOiAzODcsCiAgICAgICI1IjogOTMxNiwKICAgICAgIjYiOiAzNTI1LAogICAgICAiNyI6IDExNywKICAgICAgIjgiOiA0NCwKICAgICAgIjkiOiAwLAogICAgICAiMTAiOiAwLAogICAgICAiMTEiOiAwCiAgICB9LAogICAgIm1hc2tlZF9vcl9ub2RhdGFfY291bnQiOiAwLAogICAgInVuZXhwZWN0ZWRfdmFsdWVzIjogW10sCiAgICAiY291bnRzX3VuaXQiOiAibmF0aXZlIDIwIG0gY2VsbHMiLAogICAgImxhYmVsc19hcmVfZ3JvdW5kX3RydXRoIjogZmFsc2UKICB9LAogICJ0Y2kiOiB7CiAgICAiY3JzIjogIkVQU0c6MzI2NDAiLAogICAgIm5hdGl2ZV9yZXNvbHV0aW9uX20iOiBbCiAgICAgIDEwLjAsCiAgICAgIDEwLjAKICAgIF0sCiAgICAic291cmNlX2FmZmluZSI6IFsKICAgICAgMTAuMCwKICAgICAgMC4wLAogICAgICAzOTk5NjAuMCwKICAgICAgMC4wLAogICAgICAtMTAuMCwKICAgICAgMjgwMDAyMC4wCiAgICBdLAogICAgIm5vZGF0YV92YWx1ZSI6IG51bGwsCiAgICAid2luZG93X3NoYXBlX3Jvd3NfY29scyI6IFsKICAgICAgMjIzLAogICAgICAyNDQKICAgIF0sCiAgICAicmdiX2JhbmRfb3JkZXIiOiBbCiAgICAgIDEsCiAgICAgIDIsCiAgICAgIDMKICAgIF0sCiAgICAiZHR5cGUiOiAidWludDgiLAogICAgIm1hc2tlZF9vcl9vdXRzaWRlX2FvaV9waXhlbF9jb3VudCI6IDc1NiwKICAgICJkaXNwbGF5X29ubHkiOiB0cnVlLAogICAgInN0cmV0Y2giOiAibm9uZTsgcHJvdmlkZXIgdmFsdWVzIGRpc3BsYXllZCBhcyB1aW50OCIKICB9LAogICJvdXRwdXRzIjogWwogICAgImRhdGVkX3RjaV9kaXNwbGF5LnBuZyIsCiAgICAibmF0aXZlX3NjbF9jb250ZXh0LnBuZyIsCiAgICAiY2xhc3NfY29udGV4dF9yZXBvcnQuanNvbiIsCiAgICAibmF0aXZlX2NvbnRleHRfY2FjaGUubnB6IgogIF0sCiAgInJlZmxlY3RhbmNlX2JhbmRzX3VzZWQiOiBbXSwKICAic3BlY3RyYWxfaW5kaWNlc19jb21wdXRlZCI6IFtdLAogICJzYW1wbGluZ19zdGF0aW9ucyI6IFtdLAogICJmaWVsZF9vYnNlcnZhdGlvbnMiOiAwLAogICJpbmRlcGVuZGVudF8yMDE4X2xhYmVscyI6IDAsCiAgImhpc3RvcmljYWxfb3ZlcmxheSI6ICJOb3QgcHJvZHVjZWQgaW4gdGhpcyBpbXBsZW1lbnRhdGlvbi4iLAogICJ1bnJlc29sdmVkX2NvbnRleHQiOiBbCiAgICAiU0NMIGNsYXNzZXMgYXJlIG5vdCBmaWVsZC12YWxpZGF0ZWQgaGVyZS4iLAogICAgIlRDSSBpcyBhIHByb3ZpZGVyIHZpc3VhbGl6YXRpb24gYW5kIHN1cHBvcnRzIG5vIGNhbGlicmF0ZWQgY29sb3VyIG9yIHF1YW50aXRhdGl2ZSBpbmZlcmVuY2UuIiwKICAgICJObyBjdXJyZW50IHdhdGVyIGNoZW1pc3RyeSwgZWNvbG9naWNhbCBjaGFuZ2UsIGNvbmRpdGlvbiwgZGVwdGgsIHRpZGUsIG5hdmlnYWJpbGl0eSwgYWNjZXNzLCBvciBtaXNzaW9uIGZlYXNpYmlsaXR5IGlzIGVzdGFibGlzaGVkLiIKICBdLAogICJub3RfaW5mZXJyZWQiOiBbCiAgICAid2F0ZXJfY2hlbWlzdHJ5IiwKICAgICJjdXJyZW50X21hbmdyb3ZlX2V4dGVudCIsCiAgICAiZWNvbG9naWNhbF9jaGFuZ2UiLAogICAgImhlYWx0aF9vcl9zdHJlc3NfY2F1c2UiLAogICAgIndhdGVyX2RlcHRoX29yX3RpZGUiLAogICAgIm5hdmlnYWJpbGl0eSIsCiAgICAiZmxpZ2h0X29yX2xhbmRpbmdfcGVybWlzc2lvbiIsCiAgICAicGxhbnRpbmdfbmVlZCIsCiAgICAiZHJvbmVfZmVhc2liaWxpdHlfb3Jfc3VwZXJpb3JpdHkiLAogICAgImVjb2xvZ2ljYWxfYmVuZWZpdCIKICBdLAogICJkZXBsb3ltZW50X2F1dGhvcml6ZWQiOiBmYWxzZQp9\",\"sha256\":\"ac95159b90196ca3fb47bcd0f43aedf92ab73a3fc5c39219246b5e14d477d611\"}}")
WORK = Path(tempfile.mkdtemp(prefix='mangrove_offline_context_'))
for name, record in PAYLOADS.items():
    data = base64.b64decode(record['b64'], validate=True)
    if hashlib.sha256(data).hexdigest() != record['sha256']:
        raise ValueError('Embedded payload hash mismatch: ' + name)
    (WORK / name).write_bytes(data)
sys.path.insert(0, str(WORK))
print('Exact cached inputs restored; no EO network request.')


## 2. Replay native labels and quality masks

Checks cover identity, shapes, dtypes, native crop affines, retained membership, SCL NoData exclusion, class counts and TCI mask closure. AOI membership is reused, not independently recalculated. The calculation generates no reflectance, historical overlay, model fit or figures; it exports only the small replay-report ZIP.


In [ ]:
import json, zipfile
from offline_context_reader import run
try:
    RESULT = run((WORK / 'native_context_cache.npz').read_bytes(), (WORK / 'class_context_report.json').read_bytes())
except Exception as exc:
    RESULT = {'status':'failed_cached_replay', 'exception_type':type(exc).__name__, 'scientific_success':False}
RESULT['reader_source_sha256'] = PAYLOADS['offline_context_reader.py']['sha256']
RESULT['python_version'] = sys.version.split()[0]
print(json.dumps(RESULT, indent=2))
report_path = WORK / 'offline_replay_report.json'
report_path.write_text(json.dumps(RESULT, indent=2) + chr(10), encoding='utf-8')
# Only a NEW small replay report is exported; no repeated images/cache/source transfer.
zip_path = WORK / 'Mangrove_Pulse_Offline_Context_Replay_Report.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_path, report_path.name)
print('NEW_REPORT_ZIP_BYTES', zip_path.stat().st_size)
print('NEW_REPORT_ZIP_SHA256', hashlib.sha256(zip_path.read_bytes()).hexdigest())
from google.colab import files
files.download(str(zip_path))


## Interpretation

Successful replay establishes that the retained crop reproduces its recorded descriptive label counts. It does not independently validate the data product or imply an aircraft should be deployed. Historical cover review, contemporary photography and site-level contact-water research remain distinct. Stop if the report is not `succeeded_cached_summary_replay`; failed output is diagnostic only.

The proposed water-supported mechanism still requires an authorized, quality-matched measurement comparison against simpler methods. Conditional algebra cannot replace those observations.
